In [11]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_ollama.llms import OllamaLLM

In [ ]:
#define tool
tool = TavilySearchResults(max_results=4,tavily_api_key='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')

In [16]:
print(tool.name)

tavily_search_results_json


In [ ]:
tool

In [ ]:
#agent state define to hold list of messages
class AgentState(TypedDict):
    messages : Annotated[list[AnyMessage],operator.add]

In [ ]:
class Agent:

    def __init__(self,model,tools,system=""):
        #prompt
        self.system = system
        #define the graph
        graph = StateGraph(AgentState)
        #add first node (llm model)
        graph.add_node("llm",self.call_model)
        #add second node (take action based on llm response)
        graph.add_node("action",self.take_action)
        #check for any actions to take
        graph.add_conditional_edges("llm",self.exists_action,{True:"action",False:END})
        #connect nodes
        graph.add_edge("action","llm")
        #set starting point
        graph.set_entry_point("llm")
        #compile graph
        self.graph = graph.compile()
        #get set of tools
        self.tools = {t.name: t for t in tools}
        #bind tools with model
        self.model = model.bind_tools(tools)

    #condition for taking action
    def exists_action(self,state:AgentState):
        #take last message from agent state
        result = state['messages'][-1]
        #return if any tools mentioned in message
        return len(result.tool_calls) > 0


    def call_model(self,State:AgentState):
        #get messages from agent state
        messages = State['messages']
        #if user give prompt
        if self.system:
            #add new message to current message list
            messages = [SystemMessage(content=self.system)] + messages
        #pass to model to get response
        message = self.model.invoke(messages)
        return {'messages':[message]}

    def take_action(self,state:AgentState):
        #get any tool name mentioned in last message
        tool_calls = state['messages'][-1].tool_calls
        results = []
        #iterate each tools
        for t in tool_calls:
            print(f"Calling : {t}")
            #check that if the tool mentioned by llm is available in our tools list
            if not t['name'] in self.tools:
                print("\n ....no tools...")
                result = "bad tool name, retry"
            else:
                #pass our input to tool to get result
                result = self.tools[t['name']].invoke(t['args'])
            #append result along with tool id, name and extracted content
            results.append(ToolMessage(tool_call_id=t['id'],name=t['name'],content=str(result)))
        print("Back to model")
        return {"messages":results}

In [39]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

In [40]:
from langchain_ollama import ChatOllama

In [41]:
model = ChatOllama(model='llama3.2:latest')

In [42]:
agent = Agent(model,[tool],system=prompt)

In [32]:
from IPython.display import Image

Image(agent.graph.get_graph().draw_png())

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [46]:
messages = [HumanMessage(content="who won 1998 fifa world cup? why that team won? why that team doesnt won in 2006")]
result = agent.graph.invoke({"messages": messages})

Calling : {'name': 'tavily_search_results_json', 'args': {'query': '1998 FIFA World Cup winner analysis and reasons for their victory, but not winning in 2006'}, 'id': '9c76af37-52e6-414a-a1ec-487d03a678a1', 'type': 'tool_call'}
Back to model


In [47]:
result

{'messages': [HumanMessage(content='who won 1998 fifa world cup? why that team won? why that team doesnt won in 2006', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2025-07-26T15:01:29.128065Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4730767500, 'load_duration': 2012099500, 'prompt_eval_count': 275, 'prompt_eval_duration': 1488654500, 'eval_count': 40, 'eval_duration': 1228921200, 'model_name': 'llama3.2:latest'}, id='run--e92b4a4d-f83b-4c8c-8ced-a364ecb4dc41-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '1998 FIFA World Cup winner analysis and reasons for their victory, but not winning in 2006'}, 'id': '9c76af37-52e6-414a-a1ec-487d03a678a1', 'type': 'tool_call'}], usage_metadata={'input_tokens': 275, 'output_tokens': 40, 'total_tokens': 315}),
  ToolMessage(content='[{\'title\': \'1998 FIFA World Cup final - Wikipedia\', \'url\': \'https

In [48]:
result['messages'][-1].content

"The team that won the 1998 FIFA World Cup was France. They won the tournament by defeating Brazil 3-0 in the final, with Zinedine Zidane scoring two goals and Emmanuel Petit scoring one.\n\nFrance's victory in the 1998 World Cup was largely due to their strong team chemistry and strategy. They were led by coach Aimé Jacquet, who had a deep understanding of his players' strengths and weaknesses. The team also had a strong midfield presence, with Zidane and Didier Deschamps controlling the game.\n\nHowever, France's victory in the 2006 World Cup was not as successful. They lost to Italy in the final, with their only goal being scored by Thierry Henry in the 7th minute of the match. France was eliminated at the group stage in the 2010 World Cup and won the tournament in 2018 under coach Didier Deschamps.\n\nIn the 2006 World Cup final, France's defeat to Italy was due to several factors, including a lack of cohesion among their players, a poor performance from Zidane, who was sent off in